In [43]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

RAW = Path('raw')
CLEAN = Path('cleaned')
CLEAN.mkdir(exist_ok=True)

order_reviews = pd.read_csv(RAW/ 'olist_order_reviews_dataset.csv')

In [44]:
order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [45]:
order_reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [46]:
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53


In [47]:
order_reviews['review_score'].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [48]:
date_cols = [
    'review_creation_date',
    'review_answer_timestamp'
]

order_reviews[date_cols] = order_reviews[date_cols].apply(pd.to_datetime)

In [49]:
order_reviews['review_id'].is_unique

False

In [50]:
order_reviews['review_id'].nunique()

98410

In [51]:
order_reviews['review_id'].duplicated().sum()

np.int64(814)

In [52]:
order_reviews.duplicated(subset=['review_id','order_id']).sum()

np.int64(0)

In [53]:
(order_reviews['order_id'].value_counts() > 1).sum()

np.int64(547)

In [54]:
(order_reviews['review_answer_timestamp'] < order_reviews['review_creation_date']).sum()

np.int64(0)

In [55]:
order_reviews['is_latest_review'] = ~order_reviews.sort_values(['order_id','review_answer_timestamp']).duplicated(subset='order_id', keep='last')
order_reviews['is_latest_review'].sum()

np.int64(98673)

In [56]:
order_reviews['order_id'].nunique()

98673

In [57]:
vc = order_reviews['order_id'].value_counts()
order_reviews[order_reviews['order_id'].isin(vc[vc > 1].index)].sort_values('order_id').head(6)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,is_latest_review
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29,2017-08-30 01:59:12,True
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,"Estou acostumada a comprar produtos pelo baratheon. Super confiável, transparência, qualidade e agilidade. Super recomendo.",2017-08-25,2017-08-29 21:45:57,False
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22,2018-02-23 12:12:30,False
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04,2018-03-05 17:02:00,True
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30,2018-01-02 10:54:06,False
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30,2018-01-02 10:54:47,True


In [58]:
orders = pd.read_csv(CLEAN/ 'orders_clean.csv', usecols=['order_id','order_status'])

In [59]:
(order_reviews['order_id'].isin(orders['order_id'])).sum()

np.int64(99224)

In [60]:
(orders['order_id'].isin(order_reviews['order_id'])).sum()

np.int64(98673)

In [61]:
orders.loc[~orders['order_id'].isin(order_reviews['order_id']),'order_status'].value_counts()

order_status
delivered      646
shipped         75
canceled        20
unavailable     14
processing       6
invoiced         5
created          2
Name: count, dtype: int64

In [62]:
order_reviews['has_comment'] = order_reviews['review_comment_message'].notna()
order_reviews['has_comment'].sum()          # expect 40,977

np.int64(40977)

In [63]:
order_reviews.to_csv(CLEAN / 'order_reviews_clean.csv', index=False)

check = pd.read_csv(CLEAN / 'order_reviews_clean.csv', parse_dates=['review_creation_date', 'review_answer_timestamp'])
print(check.shape)                                                          # expect (99224, 9)
print(check.duplicated(subset=['review_id', 'order_id']).sum())             # expect 0
print(check['is_latest_review'].sum())                                      # expect 98,673
print(check['has_comment'].sum())                                           # expect 40,977
print(check['review_score'].value_counts().sort_index())                    # same as before
print(check[['review_creation_date', 'review_answer_timestamp']].dtypes)    # both datetime64

(99224, 9)
0
98673
40977
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


In [64]:
orders.head()

,order_id,order_status
0,e481f51cbdc54678b7cc49136f2d6af7,delivered
1,53cdb2fc8bc7dce0b6741e2150273451,delivered
2,47770eb9100c2d0c44946d9cf07ec65d,delivered
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered


In [65]:
order_reviews.columns.to_list()

['review_id',
 'order_id',
 'review_score',
 'review_comment_title',
 'review_comment_message',
 'review_creation_date',
 'review_answer_timestamp',
 'is_latest_review',
 'has_comment']